# Event Extraction System

**NLP IE — Identify who did what, when, and where**

## Project Overview

Extract structured events from news text:
**who** (agent), **what** (action), **when** (time), **where** (location).

## Learning Objectives

1. Extract structured events from text.
2. Combine NER with event trigger verbs.
3. Evaluate event extraction quality.

## Problem Statement

Extract WHO did WHAT, WHEN, and WHERE from news text.

## Why This Project Matters

- News monitoring needs automated extraction.
- Structured events power timelines and knowledge bases.

## Dataset Overview

8 synthetic news sentences with labeled events.

## Dataset Source and License Notes

Synthetic.

## Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ["pandas","numpy","matplotlib","seaborn","spacy"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
try:
    import spacy; spacy.load("en_core_web_sm")
except OSError:
    subprocess.check_call([sys.executable,"-m","spacy","download","en_core_web_sm"])
print("Environment ready.")


## Imports

In [ ]:
import re, warnings
from collections import defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import spacy
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
%matplotlib inline
nlp = spacy.load("en_core_web_sm")
SEED = 42; np.random.seed(SEED)
print("Loaded.")


## Configuration / Constants

In [ ]:
EVENT_VERBS = {"announced","launched","acquired","signed","opened",
    "appointed","resigned","discovered","released","reported",
    "invested","merged","filed","won","lost","visited"}
print(f"Event verbs: {len(EVENT_VERBS)}")


## Dataset Download or Loading

In [ ]:
NEWS = [
    {"text":"Apple announced a new iPhone in Cupertino on September 12, 2025.","who":"Apple","what":"announced","where":"Cupertino","when":"September 12, 2025"},
    {"text":"Google acquired DeepMind in London in 2014.","who":"Google","what":"acquired","where":"London","when":"2014"},
    {"text":"Microsoft invested $10B in OpenAI in January 2023.","who":"Microsoft","what":"invested","where":None,"when":"January 2023"},
    {"text":"Elon Musk launched SpaceX Starship from Texas on April 20, 2023.","who":"Elon Musk","what":"launched","where":"Texas","when":"April 20, 2023"},
    {"text":"The EU signed a trade deal in Brussels on March 1, 2024.","who":"The EU","what":"signed","where":"Brussels","when":"March 1, 2024"},
    {"text":"Amazon opened a fulfillment center in Nashville in 2025.","who":"Amazon","what":"opened","where":"Nashville","when":"2025"},
    {"text":"NASA discovered water on Mars and reported findings in Washington.","who":"NASA","what":"discovered","where":"Washington","when":None},
    {"text":"Tesla released Q4 earnings on January 29, 2025.","who":"Tesla","what":"released","where":None,"when":"January 29, 2025"},
]
df = pd.DataFrame(NEWS)
print(f"Loaded {len(df)} sentences")


## Data Validation Checks

In [ ]:
assert df["text"].notna().all()
print(f"Validated: {len(df)}")


## Exploratory Data Analysis

In [ ]:
df["words"] = df["text"].str.split().apply(len)
print(f"Avg words: {df["words"].mean():.0f}")


## Target Analysis

WHO, WHAT, WHEN, WHERE event components.

## Train/Validation/Test Split Strategy

Not applicable — rule-based.

## Preprocessing Strategy

NER for WHO/WHERE/WHEN. Event trigger verbs for WHAT.

## Feature Engineering — Event Extractor

In [ ]:
def extract_events(text):
    doc = nlp(text)
    agents = [e.text for e in doc.ents if e.label_ in ("PERSON","ORG","NORP")]
    locations = [e.text for e in doc.ents if e.label_ in ("GPE","LOC","FAC")]
    times = [e.text for e in doc.ents if e.label_ in ("DATE","TIME")]
    triggers = [t for t in doc if t.lemma_.lower() in EVENT_VERBS or t.text.lower() in EVENT_VERBS]
    if triggers:
        return [{"who": agents[0] if agents else None,
                 "what": triggers[0].text,
                 "where": locations[0] if locations else None,
                 "when": times[0] if times else None}]
    return []
print("extract_events() ready.")


## Baseline Model — Extract Events

In [ ]:
all_events = []
for _, row in df.iterrows():
    events = extract_events(row["text"])
    all_events.append(events)
    print(f"\n  {row["text"][:50]}...")
    for e in events:
        print(f"    WHO: {e["who"]}, WHAT: {e["what"]}, WHERE: {e["where"]}, WHEN: {e["when"]}")
df["events"] = all_events


## LazyPredict Benchmark

> **Not applicable.** NLP IE task.

## FLAML AutoML Run

> **Not applicable.** FLAML not used for NLP IE.

## Additional Best-Library Workflow — Evaluation

In [ ]:
who_ok = where_ok = when_ok = 0
n = len(df)
for _, row in df.iterrows():
    if row["events"]:
        e = row["events"][0]
        if e["who"] and row["who"] and e["who"] in row["who"]: who_ok += 1
        if row["where"] is None and e["where"] is None: where_ok += 1
        elif e["where"] and row["where"] and e["where"] in row["where"]: where_ok += 1
        if row["when"] is None and e["when"] is None: when_ok += 1
        elif e["when"] and row["when"] and e["when"] in row["when"]: when_ok += 1
print(f"WHO: {who_ok/n:.1%}, WHERE: {where_ok/n:.1%}, WHEN: {when_ok/n:.1%}")


## Top 3 Model Selection

Single event extraction pipeline.

In [ ]:
rates = {}
for f in ["who","where","when"]:
    rates[f] = sum(1 for evs in all_events if evs and evs[0][f] is not None) / len(df)
fig, ax = plt.subplots(figsize=(7,4))
ax.bar(rates.keys(), rates.values(), color="steelblue")
ax.set_title("Event Component Extraction Rate"); ax.set_ylim(0,1.1)
plt.tight_layout(); plt.show()


## Final Training and Evaluation of Top 3

NER + trigger verb pipeline is the final system.

## Error Analysis

In [ ]:
for _, row in df.iterrows():
    if not row["events"]: print(f"  No events: {row["text"][:50]}")


## Interpretation and Business Insight

1. NER + verb triggers capture basic events.
2. WHO detection is reliable via ORG/PERSON.
3. WHEN works well for explicit dates.
4. For production, use event detection models.

## Limitations

- Limited trigger verbs.
- One event per sentence.
- No event coreference.

## How to Improve This Project

1. Transformer-based event extraction.
2. Multi-event per sentence.
3. Build event timeline.
4. LLM prompting.

## Production Considerations

- Real-time news feed processing.
- Event deduplication.
- Event type monitoring.

## Common Mistakes

1. Missing unusual verb forms.
2. Wrong agent in passive voice.
3. Ignoring event negation.

## Mini Challenge / Exercises

1. Add event classification.
2. Handle passive voice.
3. Build timeline visualization.

## Final Summary / Key Takeaways

1. Event extraction structures news into records.
2. NER + dependency parsing is the simplest approach.
3. Real news requires ML models.
4. Event databases power business intelligence.